# Structured CSV Batch Fitting

This notebook shows how to fit many SWV traces from a structured dataframe or CSV file. The preferred input shape is one row per voltammogram: `voltage` is the full array of potentials, `current` is the full array of currents, and the remaining columns are metadata such as frequency, sample number, channel, and time.

The batch API fits each row and returns JSON/CSV-friendly results that can be plotted or opened in the Streamlit viewer.

For a focused example, this notebook fits channels 0 through 3 at 10 Hz and 200 Hz only.


In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "src" / "aswift").exists():
        repo_root = candidate
        break
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from aswift import (
    fit_dataframe,
    plot_signal_over_time,
)


## Required Structured DataFrame Shape

Each row should represent one complete voltammogram.

Required array columns:

- `voltage`: full one-dimensional array/list of potentials for the trace.
- `current`: full one-dimensional array/list of measured currents for the trace.

Recommended metadata columns:

- `file`: source file or trace name.
- `hz`: SWV frequency.
- `num`: sample/acquisition number, usually the time-ordered replicate index.
- `channel`: electrode/channel identifier.
- `time`: elapsed acquisition time, usually hours from the first measurement.

`fit_dataframe` treats each row as one trace when `voltage` and `current` contain arrays. The metadata columns are carried through to the results dataframe so users can sort, filter, group, and plot later. Older long-form tables with one row per sampled point are still supported for compatibility, but the array-per-trace format is the cleaner format for new data.


In [ ]:
EXAMPLE_DATA_URL = "https://github.com/Soh-Lab/aswift/releases/download/v1.0.3/example_data.zip"
examples_dir = repo_root / "examples" if (repo_root / "examples").exists() else Path.cwd().resolve()
bundled_archive = repo_root / "release_assets" / "example_data.zip"
EXAMPLE_DATA_ZIP = bundled_archive if bundled_archive.exists() else examples_dir / "example_data.zip"
STRUCTURED_PATH = examples_dir / "example_data" / "structured_csv_example" / "dox_invitro_shouldering_structured.csv"

if not STRUCTURED_PATH.exists():
    if not EXAMPLE_DATA_ZIP.exists():
        urllib.request.urlretrieve(EXAMPLE_DATA_URL, EXAMPLE_DATA_ZIP)
    with zipfile.ZipFile(EXAMPLE_DATA_ZIP) as zf:
        zf.extractall(examples_dir)

SELECTED_CHANNELS = [0, 1, 2, 3]
SELECTED_FREQUENCIES_HZ = [10, 200]

df = pd.read_csv(STRUCTURED_PATH)
df = df.loc[
    df["channel"].isin(SELECTED_CHANNELS)
    & df["hz"].isin(SELECTED_FREQUENCIES_HZ)
].reset_index(drop=True)

df.head()


In [ ]:
results = fit_dataframe(
    df,
    method="aswift",    # or "poly_linear"
    n_workers=8,
)

results[["hz", "num", "channel", "peak", "peak_voltage", "success"]].head()


## Optional Streamlit Viewer

Save the structured batch results and launch the lightweight Streamlit viewer with the JSON preloaded. The printed link opens directly on these results and the app sorts and filters by channel, frequency, and sample order.

In [ ]:
OUTPUT_DIR = Path("outputs/structured_csv_demo")
RESULTS_JSON = OUTPUT_DIR / "structured_fit_results.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_json(RESULTS_JSON, orient="records", indent=2)

import os
import socket
import subprocess
import time


def available_port(preferred=8501):
    for p in (preferred, 0):
        with socket.socket() as sock:
            try:
                sock.bind(("localhost", p))
            except OSError:
                continue
            return sock.getsockname()[1]
    raise RuntimeError("Could not find an available Streamlit port.")


import aswift.analysis.viewer_cli as aswift_viewer_cli

viewer_script = Path(str(aswift_viewer_cli.__file__)).resolve()
results_json = RESULTS_JSON.resolve()
viewer_env = os.environ.copy()
if src_root.exists():
    viewer_env["PYTHONPATH"] = os.pathsep.join(
        part for part in (str(src_root), viewer_env.get("PYTHONPATH")) if part
    )
port = available_port()
viewer_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "--server.headless=true",
        f"--server.port={port}",
        str(viewer_script),
        "--",
        str(results_json),
    ],
    cwd=repo_root,
    env=viewer_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(1)
if viewer_process.poll() is not None:
    raise RuntimeError("Streamlit failed to start. Confirm streamlit is installed in this notebook kernel.")
print(f"Saved results to {results_json}")
print(f"Streamlit viewer running at http://localhost:{port} with results loaded")
print(f"Streamlit process id: {viewer_process.pid}")

## Stop The Streamlit Viewer

Run this cell when you are done with the browser tab to stop the background Streamlit process started above.


In [ ]:
if "viewer_process" in globals() and viewer_process.poll() is None:
    viewer_process.terminate()
    try:
        viewer_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        viewer_process.kill()
        viewer_process.wait(timeout=5)
    print("Streamlit viewer stopped.")
elif "viewer_process" in globals():
    print("Streamlit viewer is already stopped.")
else:
    print("No Streamlit viewer process was started in this notebook session.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
plot_signal_over_time(results, ax=ax)
plt.show()